# Experiment 2: Document-Level Context (No CRF)

## Imports and project paths
We first import the packages needed for preprocessing, tokenization, modeling, training, and evaluation.

We also define the project root and the location of the original CoNLL-2003 raw files.

In [1]:
from pathlib import Path
import os
import numpy as np
import torch

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
import evaluate

In [2]:
from pathlib import Path
import os

def find_repo_root(start: Path = Path.cwd()):
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / "pyproject.toml").exists() or (p / ".git").exists():
            return p
    raise RuntimeError("Repo root not found")

PROJECT_ROOT = find_repo_root()
os.chdir(PROJECT_ROOT)        # optional: set CWD so other relative code works
conll_dir = PROJECT_ROOT / "data" / "conll2003"

## 1) Load and parse CoNLL-2003 files into documents
Load the raw CoNLL files and convert to document-level examples with word-level NER tags. Documents are separated by "-DOCSTART-" lines.

In [3]:
def read_conll_documents(path):
    documents = []
    current_doc_words = []
    current_doc_labels = []
    with open(str(path), encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("-DOCSTART-"):
                # Save previous document if exists
                if current_doc_words:
                    documents.append({'words': current_doc_words, 'ner_tags': current_doc_labels})
                    current_doc_words = []
                    current_doc_labels = []
                continue
            if not line:
                continue  # Skip empty lines between sentences
            # CoNLL columns: word POS CHUNK NER
            parts = line.split()
            if len(parts) >= 4:
                word = parts[0]
                ner = parts[-1]
                current_doc_words.append(word)
                current_doc_labels.append(ner)
    # Final document
    if current_doc_words:
        documents.append({'words': current_doc_words, 'ner_tags': current_doc_labels})
    return documents

# Locate files in project
train_path = conll_dir / 'eng.train'
valid_path = conll_dir / 'eng.testa'
test_path = conll_dir / 'eng.testb'

train_docs = read_conll_documents(train_path)
valid_docs = read_conll_documents(valid_path)
test_docs = read_conll_documents(test_path)

print(f"Train docs: {len(train_docs)}, Valid docs: {len(valid_docs)}, Test docs: {len(test_docs)}")
print(f"Sample doc length: {len(train_docs[0]['words'])} words")

Train docs: 946, Valid docs: 216, Test docs: 231
Sample doc length: 469 words


## 2) Create sliding windows for long documents
Documents longer than 8,192 tokens will be split into overlapping windows of 8,192 tokens with 128-token overlap.

In [4]:
model_name = 'answerdotai/ModernBERT-base'
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

MAX_LENGTH = 8192
OVERLAP = 128

def create_windows_from_document(doc_words, doc_labels):
    # Tokenize the entire document
    tokenized = tokenizer(doc_words, is_split_into_words=True, add_special_tokens=False)
    input_ids = tokenized['input_ids']
    word_ids = tokenized.word_ids()
    
    total_tokens = len(input_ids)
    if total_tokens <= MAX_LENGTH:
        # No need for windows
        return [{'input_ids': input_ids, 'word_ids': word_ids, 'labels': doc_labels}]
    
    # Create sliding windows
    windows = []
    start = 0
    while start < total_tokens:
        end = min(start + MAX_LENGTH, total_tokens)
        window_input_ids = input_ids[start:end]
        window_word_ids = word_ids[start:end]
        
        # Adjust word_ids for the window (relative to window start)
        # But actually, word_ids are absolute to the document words
        # For labels, we need to know which words are in this window
        # Find the min and max word index in this window
        word_indices_in_window = set(w for w in window_word_ids if w is not None)
        if not word_indices_in_window:
            start += MAX_LENGTH - OVERLAP
            continue
        min_word = min(word_indices_in_window)
        max_word = max(word_indices_in_window)
        window_labels = doc_labels[min_word:max_word+1]
        
        # Adjust word_ids to be relative to window_labels
        adjusted_word_ids = []
        for wid in window_word_ids:
            if wid is None:
                adjusted_word_ids.append(None)
            else:
                adjusted_word_ids.append(wid - min_word)
        
        windows.append({
            'input_ids': window_input_ids,
            'word_ids': adjusted_word_ids,
            'labels': window_labels
        })
        
        start += MAX_LENGTH - OVERLAP
    
    return windows

# Test on 10 sample documents
sample_docs = train_docs[:10]
for i, doc in enumerate(sample_docs):
    windows = create_windows_from_document(doc['words'], doc['ner_tags'])
    print(f"Doc {i}: {len(doc['words'])} words -> {len(windows)} windows")
    for j, w in enumerate(windows):
        print(f"  Window {j}: {len(w['input_ids'])} tokens, {len(w['labels'])} labels")

Doc 0: 469 words -> 1 windows
  Window 0: 652 tokens, 469 labels
Doc 1: 189 words -> 1 windows
  Window 0: 266 tokens, 189 labels
Doc 2: 239 words -> 1 windows
  Window 0: 342 tokens, 239 labels
Doc 3: 76 words -> 1 windows
  Window 0: 109 tokens, 76 labels
Doc 4: 221 words -> 1 windows
  Window 0: 323 tokens, 221 labels
Doc 5: 71 words -> 1 windows
  Window 0: 118 tokens, 71 labels
Doc 6: 117 words -> 1 windows
  Window 0: 243 tokens, 117 labels
Doc 7: 129 words -> 1 windows
  Window 0: 239 tokens, 129 labels
Doc 8: 32 words -> 1 windows
  Window 0: 50 tokens, 32 labels
Doc 9: 478 words -> 1 windows
  Window 0: 636 tokens, 478 labels


## 3) Convert to Hugging Face Dataset format

In [5]:
def to_hf_format(documents):
    examples = []
    for doc in documents:
        windows = create_windows_from_document(doc['words'], doc['ner_tags'])
        for window in windows:
            examples.append({
                'input_ids': window['input_ids'],
                'word_ids': window['word_ids'],
                'labels': window['labels']
            })
    return examples

train_ds = Dataset.from_list(to_hf_format(train_docs))
valid_ds = Dataset.from_list(to_hf_format(valid_docs))
test_ds = Dataset.from_list(to_hf_format(test_docs))

dataset = DatasetDict({'train': train_ds, 'validation': valid_ds, 'test': test_ds})
dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 946
    })
    validation: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 216
    })
    test: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 231
    })
})

## 4) Align labels to tokenized subwords

In [6]:
# Build label list from training data
all_labels = sorted({l for ex in train_ds for l in ex['labels']})
label_to_id = {l: i for i, l in enumerate(all_labels)}
id_to_label = {i: l for l, i in label_to_id.items()}

def align_labels(example):
    word_ids = example['word_ids']
    labels = example['labels']
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label_to_id[labels[word_idx]])
        else:
            aligned_labels.append(-100)
        previous_word_idx = word_idx
    example['labels'] = aligned_labels
    return example

tokenized_datasets = dataset.map(align_labels)
tokenized_datasets

Map:   0%|          | 0/946 [00:00<?, ? examples/s]

Map:   0%|          | 0/216 [00:00<?, ? examples/s]

Map:   0%|          | 0/231 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 946
    })
    validation: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 216
    })
    test: Dataset({
        features: ['input_ids', 'word_ids', 'labels'],
        num_rows: 231
    })
})

## 5) Model, trainer, and training arguments

In [7]:
num_labels = len(all_labels)
model = AutoModelForTokenClassification.from_pretrained(model_name, num_labels=num_labels, id2label=id_to_label, label2id=label_to_id)

data_collator = DataCollatorForTokenClassification(tokenizer)

metric = evaluate.load('seqeval')

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)
    true_predictions = []
    true_labels = []
    for pred, lab in zip(preds, labels):
        cur_preds = []
        cur_labels = []
        for p_i, l_i in zip(pred, lab):
            if l_i == -100:
                continue
            cur_preds.append(id_to_label[int(p_i)])
            cur_labels.append(id_to_label[int(l_i)])
        true_predictions.append(cur_preds)
        true_labels.append(cur_labels)
    results = metric.compute(predictions=true_predictions, references=true_labels)
    return { 'precision': results['overall_precision'], 'recall': results['overall_recall'], 'f1': results['overall_f1'], 'accuracy': results.get('overall_accuracy', 0) }

training_args = TrainingArguments(
    output_dir='./modernbert-doccontext',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=3e-5,
    per_device_train_batch_size=2,  # smaller due to long sequences
    per_device_eval_batch_size=2,
    num_train_epochs=5,
    weight_decay=0.01,
    fp16=True,
)

import inspect
trainer_kwargs = {
    'model': model,
    'args': training_args,
    'train_dataset': tokenized_datasets['train'],
    'eval_dataset': tokenized_datasets['validation'],
    'data_collator': data_collator,
    'compute_metrics': compute_metrics,
}
if 'tokenizer' in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Trainer(**trainer_kwargs)

trainer

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForTokenClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## 6) Train and evaluate

In [9]:
# Uncomment to run training
trainer.train()
trainer.evaluate(tokenized_datasets['test'])

print('Document-level context notebook ready. Run training on Rivanna or with sufficient GPU memory.')

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.065505,0.880705,0.899529,0.890017,0.983295
2,0.171195,0.043086,0.928821,0.937731,0.933255,0.989545
3,0.038202,0.041167,0.934866,0.944463,0.939640,0.990771


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: on_train_begin must be called before on_evaluate

## Verification of token-to-label alignment

In [10]:
# Verify alignment on 10 sample documents
sample_docs = train_docs[:10]
for i, doc in enumerate(sample_docs):
    print(f"\nDocument {i}: {len(doc['words'])} words")
    windows = create_windows_from_document(doc['words'], doc['ner_tags'])
    for j, window in enumerate(windows):
        print(f"  Window {j}: {len(window['input_ids'])} tokens")
        # Decode tokens back to words for verification
        decoded = tokenizer.decode(window['input_ids'], skip_special_tokens=True)
        print(f"    Decoded: {decoded[:100]}...")
        # Check label alignment
        word_ids = window['word_ids']
        labels = window['labels']
        aligned = []
        prev = None
        for wid in word_ids:
            if wid is None:
                aligned.append(-100)
            elif wid != prev:
                aligned.append(label_to_id[labels[wid]])
            else:
                aligned.append(-100)
            prev = wid
        print(f"    Aligned labels length: {len(aligned)}, matches tokens: {len(aligned) == len(window['input_ids'])}")
        # Spot check first few
        print(f"    First 10 aligned labels: {aligned[:10]}")


Document 0: 469 words
  Window 0: 652 tokens
    Decoded: EUrejectsGermancalltoboycottBritishlamb.PeterBlackburnBRUSSELS1996-08-22TheEuropeanCommissionsaidonT...
    Aligned labels length: 652, matches tokens: True
    First 10 aligned labels: [2, 8, -100, 1, 8, 8, 8, -100, 1, 8]

Document 1: 189 words
  Window 0: 266 tokens
    Decoded: RareHendrixsongdraftsellsforalmost$17,000.LONDON1996-08-22ArareearlyhandwrittendraftofasongbyU.S.gui...
    Aligned labels length: 266, matches tokens: True
    First 10 aligned labels: [8, -100, 3, -100, -100, 8, 8, 8, -100, 8]

Document 2: 239 words
  Window 0: 342 tokens
    Decoded: ChinasaysTaiwanspoilsatmospherefortalks.BEIJING1996-08-22ChinaonThursdayaccusedTaipeiofspoilingtheat...
    Aligned labels length: 342, matches tokens: True
    First 10 aligned labels: [0, 8, -100, 0, -100, -100, 8, -100, -100, 8]

Document 3: 76 words
  Window 0: 109 tokens
    Decoded: ChinasaystimerightforTaiwantalks.BEIJING1996-08-22Chinahassaiditwastimeforpolitic